# 🚀 Amazing-QR: 9 Adımlı Profesyonel İş Akışı v1.2

Bu notebook, **Amazing-QR** projesini 9 adımda tam kontrollü ve profesyonel bir şekilde çalıştırmak için tasarlanmıştır.

---

## 🛠️ 1. Kurulum ve Hazırlık
Sistemi hazırlar ve gerekli kütüphaneleri yükler.

In [ ]:
import os
import shutil
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

# 1. Projeyi GitHub'dan çek (Veya güncelle)
repo_name = "amazing-qr"
if not os.path.exists(repo_name):
    print("🚀 Proje indiriliyor...")
    !git clone https://github.com/orioninsist/amazing-qr.git
    %cd {repo_name}
else:
    %cd {repo_name}
    !git pull

# 2. Sistem Bağımlılıklarını Kur (ZBar)
!apt-get install -y -qq libzbar0

# 3. Python Paketlerini Kur (SABİT VERSİYONLAR)
print("📦 Paketler yükleniyor (requirements.txt baz alınıyor)...")
!pip install -q -r requirements.txt
!pip install -q -e .

# 4. WeChat QR Modellerini Otomatik Kur
if os.path.exists('setup_models.py'):
    print("🧠 Yapay Zeka modelleri kontrol ediliyor...")
    !python3 setup_models.py

# Klasör yapısını hazırla
!mkdir -p inputs/assets
!mkdir -p output

print("✅ Kurulum ve Modeller tamamlandı.")

## 📁 2. order.csv Yükle
İşlenecek QR kod listesini içeren CSV dosyasını yükleyin.

In [ ]:
csv_path = 'inputs/order.csv'

def upload_csv(b):
    clear_output()
    display(HTML("<div style='color: #2980b9; font-weight: bold;'>📁 Lütfen order.csv dosyasını seçin...</div>"))
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        with open(csv_path, 'wb') as f:
            f.write(uploaded[filename])
        display(HTML(f"<div style='color: #27ae60; font-weight: bold;'>✅ {filename} yüklendi.</div>"))
        display(HTML("<p>Şimdi Adım 3'e geçerek assetleri yükleyebilirsiniz.</p>"))

up_csv_btn = widgets.Button(description="📁 order.csv Yükle", button_style='primary', layout={'width': '250px', 'height': '50px'})
up_csv_btn.on_click(upload_csv)
display(up_csv_btn)

## 🖼️ 3. Assetleri Yükle (Logos, GIFs, Backgrounds)
QR kodlarda kullanacağınız görselleri buraya yükleyin.

In [ ]:
def upload_assets(b):
    display(HTML("<div style='color: #2980b9;'>🖼️ Lütfen arka plan/logo dosyalarını seçin (Çoklu seçim yapılabilir)...</div>"))
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest = os.path.join('inputs/assets', filename)
        with open(dest, 'wb') as f:
            f.write(uploaded[filename])
        display(HTML(f"<div style='color: #27ae60;'>✅ {filename} yüklendi.</div>"))

up_asset_btn = widgets.Button(description="🖼️ Asset Yükle", button_style='info', layout={'width': '250px', 'height': '50px'})
up_asset_btn.on_click(upload_assets)
display(up_asset_btn)

## ✍️ 4. Siparişleri Düzenle (Editor)
Tablo üzerinde toplu veya manuel değişiklikler yapın.

In [ ]:
def load_data():
    if not os.path.exists(csv_path): return pd.DataFrame()
    df = pd.read_csv(csv_path)
    cols = ['selected', 'words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name']
    for c in cols:
        if c not in df.columns:
            if c == 'selected': df[c] = True
            elif c == 'version': df[c] = 1
            elif c == 'level': df[c] = 'H'
            elif c in ['contrast', 'brightness']: df[c] = 1.0
            elif c == 'colorized': df[c] = True
            else: df[c] = ""
    return df[cols]

def save_data(df): df.to_csv(csv_path, index=False)

editor_out = widgets.Output()

def refresh_editor():
    with editor_out:
        clear_output()
        df = load_data()
        if df.empty: 
            display(HTML("<div style='padding: 20px; background: #fdf2f2; border-radius: 10px; border: 1px solid #f8d7da; color: #721c24;'>❌ Tablo bulunamadı. Lütfen Adım 2'yi kullanarak bir CSV yükleyin.</div>"))
            return
        
        bulk_btn = widgets.Button(description="🛠️ Toplu Düzenle", button_style='warning', icon='gears', layout={'width': '180px'})
        bulk_btn.on_click(lambda b: show_bulk_editor())
        
        select_all_btn = widgets.Button(description="✅ Tümünü Seç", button_style='success', icon='check-square', layout={'width': '150px'})
        select_all_btn.on_click(lambda b: toggle_all(True))
        
        deselect_all_btn = widgets.Button(description="⬜ Seçimi Kaldır", button_style='danger', icon='square-o', layout={'width': '150px'})
        deselect_all_btn.on_click(lambda b: toggle_all(False))
        
        refresh_btn = widgets.Button(description="🔄 Yenile", button_style='info', icon='refresh', layout={'width': '120px'})
        refresh_btn.on_click(lambda b: refresh_editor())
        
        display(widgets.HBox([bulk_btn, select_all_btn, deselect_all_btn, refresh_btn], layout={'gap': '10px', 'margin-bottom': '15px'}))

        header_html = \"\"\"
        <div style=\"display: flex; background: #2c3e50; color: white; padding: 10px; border-radius: 5px 5px 0 0; font-weight: bold; align-items: center;\">
            <div style=\"width: 40px; text-align: center;\">Seç</div>
            <div style=\"width: 200px; padding-left: 10px;\">URL / Metin</div>
            <div style=\"width: 40px; text-align: center;\">V</div>
            <div style=\"width: 40px; text-align: center;\">L</div>
            <div style=\"width: 150px; padding-left: 10px;\">Görsel</div>
            <div style=\"width: 100px; text-align: center;\">İşlem</div>
        </div>
        \"\"\"
        display(HTML(header_html))
        
        rows = []
        for i, row in df.iterrows():
            sel = widgets.Checkbox(value=bool(row['selected']), layout={'width': '40px'})
            def on_sel(change, idx=i): 
                d = load_data()
                d.at[idx, 'selected'] = change['new']
                save_data(d)
            sel.observe(on_sel, 'value')
            
            edit = widgets.Button(description=\"Düzenle\", icon='pencil', layout={'width': '90px'}, button_style='primary')
            edit.on_click(lambda b, idx=i: show_row_editor(idx))
            
            bg_color = \"#f8f9fa\" if i % 2 == 0 else \"#ffffff\"
            r = widgets.HBox([
                sel, 
                widgets.Label(str(row['words'])[:25] + (\"...\" if len(str(row['words'])) > 25 else \"\"), layout={'width': '200px'}), 
                widgets.Label(str(row['version']), layout={'width': '40px'}, style={'font_weight': 'bold'}),
                widgets.Label(str(row['level']), layout={'width': '40px'}),
                widgets.Label(str(row['picture'])[:15], layout={'width': '150px'}), 
                edit
            ], layout={'background': bg_color, 'padding': '5px', 'border-bottom': '1px solid #dee2e6', 'align-items': 'center'})
            rows.append(r)
        
        table_container = widgets.VBox(rows, layout={'border': '1px solid #dee2e6', 'border-top': 'none', 'border-radius': '0 0 5px 5px', 'max-height': '400px', 'overflow-y': 'auto'})
        display(table_container)

def toggle_all(value):
    df = load_data()
    df['selected'] = value
    save_data(df)
    refresh_editor()

def show_row_editor(idx):
    with editor_out:
        clear_output()
        df = load_data()
        row = df.iloc[idx]
        
        display(HTML(f\"<h3 style='color: #2980b9; border-bottom: 2px solid #2980b9; padding-bottom: 5px;'>✍️ Satır Düzenle (#{idx+1})</h3>\"))
        
        w_words = widgets.Text(value=str(row['words']), description='Words:', layout={'width': '90%'})
        w_ver = widgets.IntSlider(value=int(row['version']), min=1, max=40, description='Version:', layout={'width': '90%'})
        w_lvl = widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value=row['level'], description='Level:', layout={'width': '90%'})
        w_pic = widgets.Text(value=str(row['picture']), description='Picture:', layout={'width': '90%'})
        w_col = widgets.Checkbox(value=bool(row['colorized']), description='Colorized')
        w_con = widgets.FloatSlider(value=float(row['contrast']), min=0.1, max=3.0, step=0.1, description='Contrast:', layout={'width': '90%'})
        w_bri = widgets.FloatSlider(value=float(row['brightness']), min=0.1, max=3.0, step=0.1, description='Brightness:', layout={'width': '90%'})
        w_name = widgets.Text(value=str(row['save_name']), description='Name:', layout={'width': '90%'})
        
        save = widgets.Button(description=\"Kaydet\", button_style='success', icon='check', layout={'width': '150px'})
        cancel = widgets.Button(description=\"İptal\", button_style='danger', icon='times', layout={'width': '150px'})
        
        def on_save(b):
            df.at[idx, 'words'] = w_words.value
            df.at[idx, 'version'] = w_ver.value
            df.at[idx, 'level'] = w_lvl.value
            df.at[idx, 'picture'] = w_pic.value
            df.at[idx, 'colorized'] = w_col.value
            df.at[idx, 'contrast'] = w_con.value
            df.at[idx, 'brightness'] = w_bri.value
            df.at[idx, 'save_name'] = w_name.value
            save_data(df)
            refresh_editor()
            
        save.on_click(on_save)
        cancel.on_click(lambda b: refresh_editor())
        
        form = widgets.VBox([
            w_words, w_ver, w_lvl, w_pic, w_col, w_con, w_bri, w_name, 
            widgets.HBox([save, cancel], layout={'margin-top': '15px', 'gap': '10px'})
        ], layout={'padding': '15px', 'background': '#f8f9fa', 'border-radius': '10px', 'border': '1px solid #dee2e6'})
        display(form)

def show_bulk_editor():
    with editor_out:
        clear_output()
        df = load_data()
        
        display(HTML(\"<h3 style='color: #e67e22; border-bottom: 2px solid #e67e22; padding-bottom: 5px;'>🛠️ Toplu Düzenleme</h3>\"))
        display(HTML(\"<p style='color: #666; font-style: italic;'>Değiştirmek istediğiniz alanları seçin ve değerini girin.</p>\"))
        
        fields = {
            'words': (widgets.Checkbox(description=\"URL / Metin\"), widgets.Text(placeholder=\"https://google.com\")),
            'version': (widgets.Checkbox(description=\"Versiyon (1-40)\"), widgets.IntSlider(value=1, min=1, max=40)),
            'level': (widgets.Checkbox(description=\"Hata Düzeltme (L-H)\"), widgets.Dropdown(options=['L', 'M', 'Q', 'H'], value='H')),
            'picture': (widgets.Checkbox(description=\"Görsel (Arka Plan)\"), widgets.Text(placeholder=\"logo.png\")),
            'colorized': (widgets.Checkbox(description=\"Renklendirme\"), widgets.Checkbox(value=True)),
            'contrast': (widgets.Checkbox(description=\"Kontrast\"), widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1)),
            'brightness': (widgets.Checkbox(description=\"Parlaklık\"), widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1)),
            'save_name': (widgets.Checkbox(description=\"Dosya Adı\"), widgets.Text(placeholder=\"cikti.png\"))
        }
        
        apply_rows = []
        for key, (check, val) in fields.items():
            apply_rows.append(widgets.HBox([check, val], layout={'align_items': 'center', 'margin-bottom': '5px'}))
            
        apply_b = widgets.Button(description=\"Tümüne Uygula\", button_style='warning', icon='bolt', layout={'width': '200px'})
        cancel_b = widgets.Button(description=\"İptal\", button_style='danger', icon='times', layout={'width': '100px'})
        
        def on_apply(b):
            for i in range(len(df)):
                for key, (check, val) in fields.items():
                    if check.value:
                        df.at[i, key] = val.value
            save_data(df)
            refresh_editor()
            
        apply_b.on_click(on_apply)
        cancel_b.on_click(lambda b: refresh_editor())
        
        display(widgets.VBox(apply_rows + [widgets.HBox([apply_b, cancel_b], layout={'margin-top': '20px', 'gap': '10px'})], 
                            layout={'padding': '15px', 'background': '#fff3e0', 'border-radius': '10px', 'border': '1px solid #ffe0b2'}))

refresh_editor()
display(editor_out)

## 📊 5. İşlem Öncesi Kontrol
Sadece seçili olan satırlar gösterilir.

In [ ]:
df = load_data()
if not df.empty:
    selected_df = df[df['selected'] == True]
    display(HTML("<h3 style='color: #2c3e50;'>🚀 İşleme Alınacak Liste</h3>"))
    display(selected_df)
else:
    print("❌ Liste boş.")

## ⚙️ 6. Üretimi Başlat (Process)
QR kod üretimini başlatır.

In [ ]:
from batch_process import process_items
output_dir = 'output'
if os.path.exists(output_dir): shutil.rmtree(output_dir)
os.makedirs(output_dir)

df = load_data()
selected_data = df[df['selected'] == True].to_dict('records')
process_reports = []

print(f"🚀 {len(selected_data)} adet QR kod üretiliyor...")
for msg, path, report in process_items(selected_data, 'inputs/assets', output_dir, auto_repair=False):
    if msg: print(msg)
    if report: process_reports.append(report)

print("\n✅ Üretim tamamlandı.")

## ✨ 7. Sonuçları Önizle
Üretilen tüm görselleri burada görebilirsiniz.

In [ ]:
import base64
if os.path.exists(output_dir):
    files_list = sorted([f for f in os.listdir(output_dir) if f.lower().endswith(('.png', '.gif', '.jpg'))])
    html = '<div style="display: flex; flex-wrap: wrap; gap: 10px;">'
    for f in files_list:
        with open(os.path.join(output_dir, f), "rb") as img:
            enc = base64.b64encode(img.read()).decode('utf-8')
        mime = "image/gif" if f.endswith('.gif') else "image/png"
        html += f'<div style="text-align:center; border:1px solid #ddd; padding:5px;"><img src="data:{mime};base64,{enc}" width="150"><br><small>{f}</small></div>'
    html += '</div>'
    display(HTML(html))

## 🔍 8. Kalite Kontrol (QC)
QR kodların okunabilirliği otomatik olarak test edilir (Gelişmiş pyzbar motoru aktif).

In [ ]:
qc_html = "<table border='1' style='border-collapse: collapse; width: 100%; text-align: center;'>"
qc_html += "<tr style='background: #2c3e50; color: white;'><th>Dosya</th><th>Durum</th><th>İçerik</th></tr>"
for r in process_reports:
    status = r.get('scannable', 'Unknown')
    color = "#27ae60" if "✅" in status else "#e74c3c"
    qc_html += f"<tr><td>{r['output_file']}</td><td style='color: {color}; font-weight: bold;'>{status}</td><td>{r.get('scanned_data', '-')}</td></tr>"
qc_html += "</table>"
display(HTML(qc_html))

## 🛠️ 9. Akıllı Onarım & İndir
Hatalı kodları onarır ve sonuçları ZIP olarak paketler.

In [ ]:
failed = [r for r in process_reports if "❌" in r.get('scannable', '')]

def run_repair(b):
    global process_reports
    clear_output()
    display(HTML("⏳ Hatalı kodlar onarılıyor (High Contrast Mode)... "))
    to_repair = [r for r in process_reports if "❌" in r.get('scannable', '')]
    new_reports = []
    for msg, path, report in process_items(to_repair, 'inputs/assets', 'output', auto_repair=True):
        if msg: print(msg)
        if report: new_reports.append(report)
    for nr in new_reports:
        for i, orp in enumerate(process_reports):
            if orp['output_file'] == nr['output_file']: process_reports[i] = nr
    display(HTML("✅ Onarım tamamlandı! Lütfen Adım 8'i tekrar çalıştırın veya aşağıdan indirin."))

def download_results(b):
    import time
    zip_name = f"amazing_qr_results_{int(time.time())}.zip"
    !zip -rj {zip_name} output/
    files.download(zip_name)

if failed:
    display(HTML(f"<h4 style='color: #e67e22;'>⚠️ {len(failed)} adet QR okunamadı.</h4>"))
    repair_btn = widgets.Button(description="🛠️ Onarımı Başlat", button_style='warning')
    repair_btn.on_click(run_repair)
    display(repair_btn)
else:
    display(HTML("<h4 style='color: #27ae60;'>✅ Tüm QR kodlar başarılı!</h4>"))

dl_btn = widgets.Button(description="📥 Sonuçları ZIP İndir", button_style='success')
dl_btn.on_click(download_results)
display(dl_btn)